In [0]:
import pandas as pd
import numpy as np
from datetime import datetime

# Load the data
df = spark.table("dataanalytics.ml1.loan_book_dirty").toPandas()

# A. Fix three different date formats in application_date
def parse_mixed_dates(date_str):
    if pd.isna(date_str):
        return None
    formats = [
        '%m/%d/%Y',     # 06/14/2021
        '%Y-%m-%d',     # 2021-08-28
        '%d-%b-%Y',     # 23-Aug-2021
        '%d/%m/%Y',     # 23/03/2021
        '%m/%d/%y',     # potential 2-digit year
        '%d-%b-%y'      # potential 2-digit year
    ]
    for fmt in formats:
        try:
            return pd.to_datetime(date_str, format=fmt)
        except:
            continue
    return pd.NaT

df['application_date'] = df['application_date'].apply(parse_mixed_dates)

# B. Remove duplicate rows (keep first occurrence only)
df = df.drop_duplicates(keep='first')

# C. Standardize loan_purpose (21 variations → 7 categories)
loan_purpose_mapping = {
    'debt_consolidation': 'debt_consolidation',
    'DEBT_CONSOLIDATION': 'debt_consolidation',
    'Debt Consolidation': 'debt_consolidation',
    'debt consolidation': 'debt_consolidation',
    'home_improvement': 'home_improvement',
    'Home Improvement': 'home_improvement',
    'home improvement': 'home_improvement',
    'major_purchase': 'major_purchase',
    'Major Purchase': 'major_purchase',
    'major purchase': 'major_purchase',
    'medical': 'medical',
    'MEDICAL': 'medical',
    'Medical': 'medical',
    'other': 'other',
    'OTHER': 'other',
    'Other': 'other',
    'education': 'education',
    'Education': 'education',
    'small_business': 'small_business',
    'Small Business': 'small_business',
    'small business': 'small_business'
}
df['loan_purpose'] = df['loan_purpose'].map(loan_purpose_mapping)

# D. Standardize home_ownership (14 variations → 4 categories)
home_ownership_mapping = {
    'MORTGAGE': 'MORTGAGE',
    'mortgage': 'MORTGAGE',
    'Mortgage': 'MORTGAGE',
    'RENT': 'RENT',
    'rent': 'RENT',
    'Rent': 'RENT',
    'Renting': 'RENT',
    'OWN': 'OWN',
    'own': 'OWN',
    'Own': 'OWN',
    'Owner': 'OWN',
    'OTHER': 'OTHER',
    'other': 'OTHER',
    'Other': 'OTHER'
}
df['home_ownership'] = df['home_ownership'].map(home_ownership_mapping)

# E. Fix num_open_accounts data type (double → integer)
df['num_open_accounts'] = df['num_open_accounts'].fillna(0).astype(int)

# F. Round age to nearest whole number
df['age'] = df['age'].round(0).astype(int)

# G. Handle null values with flags and imputation

# G1. Income: create flag and impute with median
df['income_missing'] = df['annual_income'].isna()
median_income = df['annual_income'].median()
df['annual_income'] = df['annual_income'].fillna(median_income)

# G2. Delinquency history: create flag and fill with -1
df['has_delinquency_history'] = df['months_since_last_delinquency'].notna()
df['months_since_last_delinquency'] = df['months_since_last_delinquency'].fillna(-1)

# G2b. Create categorical version of delinquency history
def categorize_delinquency(months):
    if months == -1:
        return 'No_History'
    elif months <= 12:
        return 'Recent_0-12mo'
    elif months <= 36:
        return 'Moderate_12-36mo'
    else:
        return 'Old_36mo+'

df['delinquency_category'] = df['months_since_last_delinquency'].apply(categorize_delinquency)

# G3. Employment: create flag and impute with 0
df['employment_missing'] = df['employment_length_years'].isna()
df['employment_length_years'] = df['employment_length_years'].fillna(0)

# G4. Num accounts: create flag and impute with median (already handled in E)
df['num_accounts_missing'] = False  # Already imputed in step E

# H. Ensure consistent month columns are numeric (already numeric)
# months_since_oldest_account, months_since_last_delinquency, months_at_current_address
# These are already in proper format

# I. High cardinality - branch_code_id (keep as is, can be addressed in modeling)
# No action needed unless specific encoding is requested

# J. Late 2023 loans observation window (flag for potential filtering in analysis)
df['incomplete_observation_window'] = df['application_date'] >= pd.Timestamp('2023-11-01')

# K. Class imbalance (note for modeling, no transformation here)
# This is noted but typically handled during modeling phase

# L. Outliers in continuous variables (note for modeling)
# Can be addressed in feature engineering or modeling phase

# M. application_date data type fixed in step A

# Reorder columns to put flags together
flag_cols = ['income_missing', 'has_delinquency_history', 'employment_missing', 
             'num_accounts_missing', 'incomplete_observation_window']
other_cols = [col for col in df.columns if col not in flag_cols and col != 'delinquency_category']
df = df[other_cols + ['delinquency_category'] + flag_cols]

In [ ]:
# Convert back to Spark DataFrame and save to silver table
spark_df = spark.createDataFrame(df)

# Save as a Delta table in the silver layer
spark_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("dataanalytics.ml1.loan_book_silver")

# Display confirmation
display(spark.sql("SELECT COUNT(*) as row_count FROM dataanalytics.ml1.loan_book_silver"))

In [ ]:
# Display the cleaned data
display(df)